# PlantAG re-training on the cleaned dataset

This notebook re-runs the PlantAG pipeline of `test1_tylcv_7_24dpi` on `adata_combined_squidpy_clean.h5ad` (the data
without the displaced material under M7-1). Every step and setting is copied from the original notebook; the only changes are:

* paths are set once in cell 2, and the outputs get new names so that the original files are not overwritten;
* random seeds are fixed (the original run fixed none), so the result can be reproduced exactly;
* the training-label cell imports `DataLoader` from `torch_geometric.loader` (the old location no longer exists);
* the Wasserstein matrix and the scran size factors are not recomputed: the original notebook computes both but PlantAG
  uses neither;
* a final cell compares the new labels with the previous PlantAG run.

Run all cells in order. On a CPU runtime the whole notebook takes roughly 15–30 minutes (the Pearson residuals and the
persistence images are the slow steps). Then point `ANN_H5AD` in the audit notebook to `OUT_FINAL`.

In [ ]:
# 0 | Colab set-up (from the original notebook) and Google Drive
!pip -q install scanpy anndata torch_geometric igraph leidenalg gudhi persim
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 1 | Paths and settings
from pathlib import Path
BASE = Path("/content/drive/MyDrive/spatial_transcriptomics")
IN_H5AD = BASE / "processed/adata_combined_squidpy_clean.h5ad"                                   # written by audit cell 3c
OUT_LABELS = BASE / "processed/adata_combined_squidpy_clean_tda_features_plantag.h5ad"           # after training labels + GCN
OUT_FINAL = BASE / "processed/adata_combined_squidpy_clean_tda_features_plantag_gnn_embedding.h5ad"   # -> ANN_H5AD in the audit
JSON_DIR = BASE / "json_data"                                                                     # trichome / chloroplast lists
MODEL_OUT = BASE / "model/plant_cell_gnn_clean.pt"
PREVIOUS_ANN = BASE / "processed/adata_combined_squidpy_tda_features_plantag_gnn_embedding.h5ad"  # previous run, for comparison
SEED = 0            # fixed seeds (new); every other setting below is the original one
N_EPOCHS = 100      # original: 100 epochs, Adam lr 0.005, weight decay 1e-4, hidden 256, dropout 0.5
DEVICE = "cpu"      # CPU gives bit-for-bit reproducible training; "cuda" is faster but not exactly reproducible
THIN_TO_MEDIAN_UMIS = None   # depth-matched sensitivity run: e.g. 100 thins every capture area to a median of 100 UMIs/spot
if THIN_TO_MEDIAN_UMIS:      # outputs get a _depth<N> suffix; the comparison in the last cell is with the full-depth model
    sfx = f"_depth{THIN_TO_MEDIAN_UMIS}"
    PREVIOUS_ANN = OUT_FINAL
    OUT_LABELS, OUT_FINAL = (p.with_name(p.name.replace(".h5ad", sfx + ".h5ad")) for p in (OUT_LABELS, OUT_FINAL))
    MODEL_OUT = MODEL_OUT.with_name(MODEL_OUT.stem + sfx + ".pt")

In [ ]:
# 2 | Imports and fixed seeds
import json, pickle, random, warnings
import numpy as np, pandas as pd, scanpy as sc, anndata as ad
import torch
warnings.filterwarnings("ignore")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
print("torch", torch.__version__, "| scanpy", sc.__version__)

In [ ]:
# 3 | Load the cleaned data; drop spots without counts (original cells 2, 4-6)
adata = sc.read_h5ad(IN_H5AD)
library_sizes = np.array(adata.X.sum(axis=1)).flatten()
non_zero_mask = library_sizes > 0
adata = adata[non_zero_mask, :].copy()
print(f"{adata.n_obs} spots kept, {int((~non_zero_mask).sum())} without counts removed")
if THIN_TO_MEDIAN_UMIS:   # binomial thinning of each capture area to the target median UMIs per spot (never upwards)
    import scipy.sparse as sp
    rng = np.random.default_rng(SEED)
    Xc = sp.csr_matrix(adata.X).tocoo()
    sec = adata.obs["sample"].astype(str).to_numpy()
    med = pd.Series(np.asarray(adata.X.sum(axis=1)).ravel()).groupby(sec).median()
    fac = np.minimum(1.0, THIN_TO_MEDIAN_UMIS / med.reindex(sec).to_numpy())
    adata.X = sp.csr_matrix((rng.binomial(np.rint(Xc.data).astype(np.int64), fac[Xc.row]).astype(np.float32), (Xc.row, Xc.col)),
                            shape=adata.shape)
    adata.X.eliminate_zeros()
    adata = adata[np.asarray(adata.X.sum(axis=1)).ravel() > 0].copy()
    after = pd.Series(np.asarray(adata.X.sum(axis=1)).ravel()).groupby(adata.obs["sample"].astype(str).to_numpy()).median()
    print(pd.DataFrame({"median_umis_before": med, "factor": np.minimum(1.0, THIN_TO_MEDIAN_UMIS / med), "median_umis_after": after}).round(2))
adata.obs["total_counts"] = np.asarray(adata.X.sum(axis=1)).ravel()
print(adata.obs["sample"].value_counts() if "sample" in adata.obs else adata)

In [ ]:
# 4 | Shifted-logarithm normalisation (original cell 8)
scales_counts = sc.pp.normalize_total(adata, target_sum=None, inplace=False)
adata.layers["log1p_norm"] = sc.pp.log1p(scales_counts["X"], copy=True) #scales_counts[X] est un dictionnaire contenant les données normalisée par cellule

In [ ]:
# 5 | Analytic Pearson residuals and the 4,000 most deviant genes (original cells 14, 23, 27)
#import des fonctions pour manipuler des matrices creuses (sparse matrices),
from scipy.sparse import csr_matrix, issparse

# Calculate analytic pearson residuals
def pearson_residuals(counts, theta, clipping=True):
    '''Computes analytical residuals for NB model with a fixed theta, clipping outlier residuals to sqrt(N)'''
    counts_sum0 = np.sum(counts, axis=0)
    counts_sum1 = np.sum(counts, axis=1)
    counts_sum  = np.sum(counts)

    #get residuals
    mu = counts_sum1 @ counts_sum0 / counts_sum
    z = (counts - mu) / np.sqrt(mu + mu/theta + 0.01)

    #clip to sqrt(n)
    if clipping:
        n = counts.shape[0]
        z[z >  np.sqrt(n)] =  np.sqrt(n)
        z[z < -np.sqrt(n)] = -np.sqrt(n)

    return z

adata.layers["analytic_pearson_residuals"] = csr_matrix(pearson_residuals(adata.X,100))

import numpy as np

# On récupère la matrice des résidus
X_resid = adata.layers["analytic_pearson_residuals"]

# Calculer l'énergie (somme des carrés) pour chaque gène (par colonne)
if hasattr(X_resid, "toarray"):
    X_resid = X_resid.toarray()  # convertir si c’est une matrice sparse

gene_scores = np.sum(X_resid ** 2, axis=0)  # shape: (n_genes,)

# Sélectionner les indices des gènes les plus "déviants"
n_top = 4000
idx = np.argsort(gene_scores)[-n_top:]  # top 4000

# Marquer dans adata.var
mask = np.zeros(adata.n_vars, dtype=bool)
mask[idx] = True
adata.var["highly_deviant"] = mask
adata.var["deviance_score"] = gene_scores

In [ ]:
# 6 | log1p layer as X, deviant genes as highly variable, PCA (original cells 29-30)
# on copie les données normalisées et log-transformées (qui sont dans la couche "log1p_norm") dans la matrice principale adata.X.
adata.X = adata.layers["log1p_norm"]
# Analyse en Composantes Principales (PCA) sur tes données dans adata
# setting highly variable as highly deviant to use scanpy 'use_highly_variable' argument in sc.pp.pca
adata.var["highly_variable"] = adata.var["highly_deviant"]
try:
    sc.pp.pca(adata, svd_solver="arpack", mask_var="highly_variable")       # current scanpy
except TypeError:
    sc.pp.pca(adata, svd_solver="arpack", use_highly_variable=True)  # older scanpy (original call)
print("X_pca:", adata.obsm["X_pca"].shape)

In [ ]:
# 7 | TDA functions (original cells 43, 45, 46, 48, verbatim)
import numpy as np
import gudhi as gd
from persim import PersistenceImager, wasserstein, bottleneck
from sklearn.neighbors import NearestNeighbors
from scipy.spatial.distance import pdist, squareform
from sklearn.preprocessing import StandardScaler

def compute_local_diagrams(X, n_neighbors=30, max_dim=2):
    """
    Calcule les diagrammes de persistance locaux pour chaque point de X.

    X : ndarray, shape (n_samples, n_features)
        Les données à analyser.
    n_neighbors : int
        Nombre de voisins pour construire le diagramme local autour de chaque point.
    max_dim : int
        Dimension maximale des complexes simpliciaux.

    Retour :
    diagrams : list
        Liste des diagrammes de persistance pour chaque point.
    """

    # Construction du voisinage pour chaque point
    nbrs = NearestNeighbors(n_neighbors=n_neighbors, algorithm='ball_tree').fit(X)
    distances, indices = nbrs.kneighbors(X)  # distances et indices des voisins

    diagrams = []  # Liste pour stocker les diagrammes

    # Boucle sur chaque point pour calculer le diagramme local
    for i in range(X.shape[0]):
        neighbors_idx = indices[i]       # indices des voisins du point i
        points = X[neighbors_idx, :]    # coordonnées des voisins
        rips = gd.RipsComplex(points=points, max_edge_length=np.max(distances[i]))  # complexe de Rips
        st = rips.create_simplex_tree(max_dimension=max_dim)  # création de l'arbre simplicial
        st.compute_persistence()         # calcul de la persistance
        diagrams.append(st.persistence())  # ajout du diagramme à la liste

    return diagrams  # retourne la liste des diagrammes

def diagrams_to_images(diagrams_clean, pixel_size=0.1):
    """
    Transforme une liste de diagrammes de persistance en images (grilles de pixels).

    diagrams_clean : list
        Liste des diagrammes de persistance nettoyés.
    pixel_size : float
        Taille d'un pixel dans l'image de persistance.

    Retour :
    images : list
        Liste des images normalisées correspondant aux diagrammes.
    """

    # Création de l'objet PersistenceImager avec la taille de pixel choisie
    pimgr = PersistenceImager(pixel_size=pixel_size)

    # Apprentissage de la “grille” pour transformer les diagrammes en images
    pimgr.fit(diagrams_clean)

    # Transformation de chaque diagramme en image
    images = [pimgr.transform(d) for d in diagrams_clean]

    # Normalisation des images pour que les valeurs soient entre 0 et 1
    # Ajout de 1e-8 pour éviter la division par zéro
    images = [img / (img.max() + 1e-8) for img in images]

    return images  # retourne la liste d'images normalisées

def diagrams_birth_to_death(diagrams_clean):
    """
    Convertit des diagrammes de persistance de format (birth, lifetime)
    en format (birth, death).

    diagrams_clean : list
        Liste de diagrammes de persistance (birth, lifetime).

    Retour :
    diagrams_bd : list
        Liste de diagrammes de persistance convertis en (birth, death).
    """

    diagrams_bd = []  # Liste pour stocker les diagrammes convertis

    # Boucle sur chaque diagramme
    for diag in diagrams_clean:
        # Cas particulier : diagramme vide → ajout d'un point artificiel
        if len(diag) == 0:
            diagrams_bd.append(np.array([[0.0, 1e-6]]))
            continue

        # Extraction des naissances et durées
        births = diag[:, 0]                 # colonnes de naissance
        deaths = diag[:, 0] + diag[:, 1]    # calcul des décès = naissance + durée

        # S'assure que death > birth pour tous les points
        deaths = np.maximum(deaths, births + 1e-8)

        # Création du diagramme (birth, death) et ajout à la liste
        diagrams_bd.append(np.column_stack([births, deaths]))

    return diagrams_bd  # retourne la liste des diagrammes convertis

import numpy as np
import gudhi as gd
from sklearn.preprocessing import StandardScaler

# --- 1. Fonctions utilitaires TDA ---

def clean_diagrams(diagrams):
    """
    Nettoie les diagrammes de persistance : remplace les valeurs infinies et convertit birth→death.
    """
    cleaned = []
    for diag in diagrams:
        # Récupère les valeurs finies de death pour calculer un max
        finite_vals = [d for _, (b, d) in diag if np.isfinite(d)]
        max_val = max(finite_vals) if finite_vals else 1.0
        points = []
        for _, (b, d) in diag:
            if not np.isfinite(d):
                d = b + max_val  # remplace l'infini par une valeur finie
            if d > b:
                points.append([b, d-b])  # conversion en (birth, lifetime)
        cleaned.append(np.array(points))
    return cleaned



# --- 2. Superlevel filtration pour images ---

def superlevel_filtration(img, n_levels=5, mode="quantile"):
    """
    Crée des images binaires filtrées par seuils adaptatifs ou linéaires.
    Chaque niveau correspond à un seuil pour extraire la topologie.
    """
    if mode == "linear":
        thresholds = np.linspace(0, 1, n_levels)
    elif mode == "quantile":
        thresholds = np.quantile(img, np.linspace(0, 1, n_levels))
    else:
        raise ValueError("mode doit être 'linear' ou 'quantile'")
    return [(t, (img >= t).astype(int)) for t in thresholds]

def persistence_from_image(img):
    """
    Calcule le diagramme de persistance d'une image via CubicalComplex.
    """
    cc = gd.CubicalComplex(dimensions=img.shape, top_dimensional_cells=img.flatten())
    diag = cc.persistence()
    return diag

def features_from_image(img):
    """
    Extrait des features simples depuis le diagramme de persistance d'une image.
    """
    diag = persistence_from_image(img)
    points = np.array([pt for _, pt in diag if np.isfinite(pt[1])])
    if points.size == 0:
        return np.zeros(8)

    births, deaths = points[:,0], points[:,1]
    pers = deaths - births
    pmax = pers.max() if len(pers) else 0.0
    eps = 1e-3 if pmax == 0 else 0.01 * pmax  # seuil pour séparer points persistants vs quasi-nuls
    n_points = len(points)
    n_diag = np.sum(pers < eps)
    n_persistent = np.sum(pers >= eps)
    total_persistence = pers.sum()
    max_persistence = pers.max()
    mean_persistence = pers.mean()
    mean_birth = births.mean()
    mean_death = deaths.mean()
    return np.array([
        n_points, n_diag, n_persistent,
        total_persistence, max_persistence,
        mean_persistence, mean_birth, mean_death
    ])

def tda_features_from_images(images, n_levels=5, mode="quantile"):
    """
    Pipeline complet pour extraire les features TDA à partir des images de persistance.
    Combine les features de plusieurs niveaux de superlevel filtration.
    """
    all_features = []
    for img in images:
        filtrations = superlevel_filtration(img, n_levels=n_levels, mode=mode)
        feats_levels = [features_from_image(fimg) for _, fimg in filtrations]
        feats_concat = np.concatenate(feats_levels)  # concatène les features de tous les niveaux
        all_features.append(feats_concat)
    return np.array(all_features)

# --- 3. Pipeline final sur AnnData ---

def compute_final_tda_features_filtered_only(adata, images, n_levels=5):
    """
    Extrait uniquement les features TDA issues des images filtrées (robustes).
    Résultat stocké dans adata.obsm["tda_features_final"].
    """
    # Features à partir des images de persistance
    tda_feat_img = tda_features_from_images(images, n_levels=n_levels, mode="quantile")  # (n_cells, n_levels*8)

    # Normalisation
    scaler = StandardScaler()
    tda_feat_img_scaled = scaler.fit_transform(tda_feat_img)

    # Stockage dans AnnData
    adata.obsm["tda_features_final"] = tda_feat_img_scaled
    return adata

In [ ]:
# 8 | Local persistence diagrams on X_pca (all components, 30 points), persistence images, cubical features
#     (original cells 49, 51, 60; the original cached images/diagrams in pickles, written here with a _clean suffix)
X = adata.obsm['X_pca']#[:, :10]
diagrams = compute_local_diagrams(X)
diagrams_clean = clean_diagrams(diagrams)
images = diagrams_to_images(diagrams_clean)
diagrams_bd = diagrams_birth_to_death(diagrams_clean)
adata = compute_final_tda_features_filtered_only(adata, images, n_levels=5)
images_vec1 = np.array([img.ravel() for img in images])
adata.obsm['images_vec1'] = images_vec1
with open(BASE / "processed/images_clean.pkl", "wb") as f:
    pickle.dump(images, f)
with open(BASE / "processed/diagrams_bd_clean.pkl", "wb") as f:
    pickle.dump(diagrams_bd, f)
print("images_vec1:", images_vec1.shape, "| tda_features_final:", adata.obsm["tda_features_final"].shape)

In [ ]:
# 9 | Marker lists (original cells 68, 74-77, 80, verbatim; JSON files read from JSON_DIR)
adata.var_names_make_unique()
gene_names = adata.var_names

import json
# Specify the path to your .jsonl file
file_path = str(JSON_DIR / "trichome.jsonl")

# Create an empty list to store the data
data_list = []

# Read the file line by line
with open(file_path, 'r') as file:
    for line in file:
        # Parse each line as JSON
        data = json.loads(line)
        # Append the data to the list
        data_list.append(data)

# Convert the list of dictionaries into a pandas DataFrame
df_3 = pd.DataFrame(data_list)
df_tric = df_3["symbol"]

trichomes = df_tric.tolist()

import json
import pandas as pd

# Specify the path to your .jsonl file
file_path = str(JSON_DIR / "chlorop_random.jsonl")

# Create an empty list to store the data
data_list = []

# Read the file line by line
with open(file_path, 'r') as file:
    for line in file:
        # Parse each line as JSON (these are simple strings, not objects)
        data = json.loads(line)
        data_list.append(data)

# Convert the list to a pandas DataFrame
df_4 = pd.DataFrame(data_list, columns=['symbol'])

# Now you can access the symbol column
df_chlorop = df_4["symbol"]

print("Success! Gene symbols extracted:")
print(df_chlorop.head())
print(f"Total genes: {len(df_chlorop)}")

Chloroplast = df_chlorop.tolist()

marker_genes = {
    'Epidermal': ['LTP3', 'LOC101267793', 'LOC101264168', 'THM6', 'LOC101253907',
                        'CER6', 'LOC101253447', 'LOC101253245', 'LOC101260448',
                        'LOC101243909', 'LOC101268888', 'GPAT', 'DFR', 'LOC101245496',
                        'AnthOMT', 'ARF3', 'NAM1', 'LOC101253573', 'LOC101245916',
                        'LOC101266023','LOC101247519', 'LOC101248013', 'LOC101261641',
                        'LOC104649377', 'LOC104644846', 'LOC101264524', 'LOC100191129'], # LOC100191129
    #'Palisade mesophyll cells': ['CAB5', 'RBCS1A', 'RBCS3B', 'RBCS2B', 'AnthOMT', 'SP5G'],
    #'Spongy mesophyll cells': ['CAB1R', 'CAB2R', 'CAB3R'],
    'Mesophyll': ['CAB5', 'RBCS1A', 'RBCS3B', 'RBCS2B', 'AnthOMT', 'SP5G',
                        'CAB1R', 'CAB2R', 'CAB3R', 'LOC10126558', 'LOC101256154', 'LOC101250583',
                        'LOC101258923', 'LOC101260653', 'LOC101248225', 'OPR1', 'LOC101254262', 'LOC101248387',
                        'LOC101252539', 'LOC101251107', 'LOC101262653', 'LOC101259432', 'LOC101259147',
                        'LOC104649338', 'LOC101262529', 'LOC101265482'], # LOC10126558
    'Guard': ['SLAH3', 'LOC101266554', 'LOC101261097', 'LOC101260348', 'KAT1',
                    'LOC101266436', 'LOC101246498', 'OST1', 'LOC101264927',
                    'LOC101250991', 'LOC101256333', 'LOC101249341', 'LOC101246125',
                    'LOC101261813', 'LOC101266341', 'LOC101256498', 'LOC101251105',
                    'LOC101251395', 'LOC109120668', 'LOC101266242', 'LOC101256903',
                    'LOC101263046', 'LOC101265367', 'LOC101254837', 'LOC101262576',
                    'LOC101252163', 'LOC101264505', 'LOC101266019', 'LOC104644323', 'LOC101263994'], # LOC101266019
    'Trichomes': trichomes + ['GL1', 'THM27', 'LOC101264905' 'WER', 'TTG1', 'LOC101262902', 'LOC101252304',
                              'LOC101262489'], # keep deepening GL1! LOC101252304
    'Vascular bundle': ['LHW', 'LOC101264638', 'LOC101248073', 'WOX4', 'LOC100301933', 'DFR',
                              'LOC101257108', 'LOC101251397', 'LOC101247402', 'LOC101251065', 'LOC101263073'],
    'Xylem': ['VND7', 'LOC101267143', 'LOC101248904', 'LOC101247411',
                    'LOC101245609', 'LOC101252786', 'LOC101246350', 'LOC101263643',
                    'IRX3', 'CESA4', 'XSP10', 'psi14C', 'psi14A', 'psi14B'],
    'Phloem': ['SUC2', 'LeSUT1', 'LOC101266694', 'LOC101264612', 'SNAT',
                     'LOC101246577','LOC101246577', 'LOC109118706', 'LOC101254356',
                     'LOC104645913', 'LOC101268738', 'LOC104645912', 'LOC101267180',
                     'LOC101254380', "LOC101266616", 'LOC101245240', 'LOC101260710'],
    'Stomatal': ['FAMA', 'LOC101246250' 'MUTE', 'LOC101246861', 'SCRM',
                       'LOC101266110', 'LOC104646598', 'LOC101264685', 'LOC101257435',
                       'LOC101254460', 'LOC101268195', 'RBR1'],
    'Parenchyma': ['NIA1', 'NIA2', 'PR1b1', 'PR1a2', 'LOC101245143', 'LOC101267404', 'LOC101248419',
                         'LOC101249757', 'Acx1A','Acx1B'], # LOC101245143
    'Bundle sheath': ['RBCS1A', 'LOC101257875' 'RBCS3B', 'LOC101257875',
                            'RBCS2B', 'LOC101253838', 'LOC101258532'], #LOC101253838
    'Subsidiary': ['TMM', 'cdkB1', 'CYCB1;1', 'LOC101248088', 'LOC101265619',
                         'LOC101258748', 'LOC101245600', 'LOC101244366', 'LOC101258426', 'TCP21'], # 'LOC101245600'
    #'Epidermal hair cells': ['WER', 'GL1', 'TTG1', 'ARF3'],
    'Hydathodes': ['SWEET9', 'LOC101259076', 'AQPs', 'NIP51', 'LOC101244210', 'LOC101265170'],
    'Vein': ['ATHB-8', 'LOC101257548', 'LOC101251205', 'KNAT1', 'LOC101254290',
                   'LOC543604', 'LBD30', 'LOC101256138'],
    'Mesophyll-Ch': Chloroplast + ['LyesC2p073', 'LyesC2p057', 'LyesC2p044', 'LyesC2p026', 'LyesC2p025',
                                'LyesC2p009', 'LyesC2p005', 'LyesC2p004', 'LOC101245517', 'LOC101268314',
                                'LOC101244690', 'LOC101254329', 'LOC101253991', 'LOC101244922', 'LOC104644275',
                                'LOC101263642', 'LOC101247998', 'LOC101256511' ]# LOC101245517

    # Add more cell types and their marker genes as needed
}

In [ ]:
# 10 | Training labels: marker scores, PCA of the persistence images, Leiden (res 1.5), cluster argmax (original cell 81)
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.sparse import issparse
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv


# 1. Préparer les scores à partir des gènes marqueurs
# On ne garde que les gènes marqueurs présents dans l'adata
marker_genes_in_data = {ct: [g for g in gs if g in adata.var_names]
                        for ct, gs in marker_genes.items()}

# Initialisation d'un DataFrame pour stocker les scores par cellule et par type de cellule
cell_scores = pd.DataFrame(index=adata.obs_names,
                           columns=marker_genes_in_data.keys(),
                           dtype=float)

# Calcul des scores : moyenne z-score des gènes marqueurs pour chaque cellule
for ct, genes in marker_genes_in_data.items():
    if not genes:  # si aucun gène marqueur disponible pour ce type
        continue
    expr = adata[:, genes].X  # récupération de l'expression des gènes
    if issparse(expr):        # si la matrice est creuse
        expr = expr.toarray() # conversion en dense
    expr_z = (expr - expr.mean(axis=0)) / expr.std(axis=0)  # z-score par gène
    expr_z = np.nan_to_num(expr_z)  # remplace NaN ou inf par 0
    cell_scores[ct] = expr_z.mean(axis=1)  # score moyen pour chaque cellule

# 2. Réduction de dimension sur les images aplaties avec PCA
X_tda_scaled = StandardScaler().fit_transform(adata.obsm['images_vec1'])  # normalisation
pca = PCA(n_components=50, random_state=SEED)  # on garde 50 composantes principales (seed fixed)
X_tda_pca = pca.fit_transform(X_tda_scaled)    # application de la PCA
adata.obsm['X_tda_pca'] = X_tda_pca           # stockage dans AnnData

# 3. Clustering avec k-NN + Leiden
sc.pp.neighbors(adata, use_rep='X_tda_pca', n_neighbors=30)  # calcul du graphe de voisins
sc.tl.leiden(adata, resolution=1.5, key_added='tda_leiden')   # clustering Leiden

# 4. Création des pseudo-labels à partir du clustering
cluster_labels = {}
for cl in adata.obs['tda_leiden'].cat.categories:
    cells = adata.obs['tda_leiden'] == cl         # cellules du cluster
    mean_scores = cell_scores.loc[cells].mean(axis=0)  # score moyen par type de cellule
    cluster_labels[cl] = mean_scores.idxmax()     # type avec score moyen maximal

# Attribution des pseudo-labels dans AnnData
adata.obs['pseudo_cell_type'] = adata.obs['tda_leiden'].map(cluster_labels)

# Vérification : compte des pseudo-labels par type
print(adata.obs['pseudo_cell_type'].value_counts())

In [ ]:
# 11 | Weighted k-NN graph and GCN training (original cell 87; seeds fixed, CPU by default)
torch.set_num_threads(max(1, torch.get_num_threads()))
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.sparse import issparse
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader  # Changed import path
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# -------------------------------
# Construire le graphe k-NN pondéré en utilisant HVG + images aplaties + TDA features
# -------------------------------
def build_weighted_graph(adata, n_neighbors=30, n_hvg_pca=50):
    # --- 1️⃣ HVG ---
    hvg = adata.var_names[adata.var['highly_variable']].tolist()
    if issparse(adata.X):
        X_hvg = adata[:, hvg].X.toarray()
    else:
        X_hvg = adata[:, hvg].X

    # Réduction dimensionnelle HVG avec PCA pour limiter le bruit
    X_hvg_scaled = StandardScaler().fit_transform(X_hvg)
    pca = PCA(n_components=min(n_hvg_pca, X_hvg_scaled.shape[1]), random_state=SEED)   # seed fixed
    X_hvg_pca = pca.fit_transform(X_hvg_scaled)

    # --- 2️⃣ Concaténation avec images aplaties et TDA features finales ---
    X_aug = np.hstack([X_hvg_pca, adata.obsm['images_vec1'], adata.obsm['tda_features_final']])
    X_aug = StandardScaler().fit_transform(X_aug)

    # --- 3️⃣ k-NN graph ---
    nbrs = NearestNeighbors(n_neighbors=n_neighbors, metric='cosine').fit(X_aug)
    distances, indices = nbrs.kneighbors(X_aug)

    edge_index = []
    edge_weight = []
    for i in range(len(indices)):
        for j_idx, j in enumerate(indices[i]):
            edge_index.append([i, j])
            edge_weight.append(1.0 - distances[i][j_idx])  # cosine similarity

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_weight = torch.tensor(edge_weight, dtype=torch.float)

    # --- 4️⃣ Labels ---
    y = pd.Categorical(adata.obs['pseudo_cell_type']).codes
    y = torch.tensor(y, dtype=torch.long)

    # --- 5️⃣ Coordonnées spatiales (optionnel) ---
    pos = torch.tensor(adata.obsm['spatial'], dtype=torch.float) if 'spatial' in adata.obsm else None

    return Data(
        x=torch.tensor(X_aug, dtype=torch.float),
        edge_index=edge_index,
        edge_weight=edge_weight,
        y=y,
        pos=pos
    )

# Construire le graphe
graph_data1 = build_weighted_graph(adata)

# -------------------------------
# Encodage labels + masques train/test
# -------------------------------
encoded_labels = graph_data1.y
num_classes = len(np.unique(encoded_labels))

train_idx, test_idx = train_test_split(np.arange(len(encoded_labels)),
                                       test_size=0.2,
                                       stratify=encoded_labels,
                                       random_state=42)
train_mask = torch.zeros(len(encoded_labels), dtype=torch.bool)
train_mask[train_idx] = True
test_mask = torch.zeros(len(encoded_labels), dtype=torch.bool)
test_mask[test_idx] = True

class_weights = compute_class_weight(class_weight='balanced',
                                     classes=np.unique(encoded_labels),
                                     y=encoded_labels.numpy())
class_weights = torch.tensor(class_weights, dtype=torch.float)
criterion = nn.CrossEntropyLoss(weight=class_weights)

# -------------------------------
# Définition du GNN
# -------------------------------
class PlantCellGNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.dropout = dropout
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, data):
        x, edge_index, edge_weight = data.x, data.edge_index, getattr(data, 'edge_weight', None)
        x = F.relu(self.conv1(x, edge_index, edge_weight))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index, edge_weight))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv3(x, edge_index, edge_weight))
        return self.classifier(x)

# Initialiser le modèle
torch.manual_seed(SEED)                                               # seed fixed
model = PlantCellGNN(input_dim=graph_data1.x.shape[1], hidden_dim=256, num_classes=num_classes, dropout=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

# -------------------------------
# DataLoader PyG (avec import corrigé)
# -------------------------------
train_loader = DataLoader([graph_data1], batch_size=1)
test_loader = DataLoader([graph_data1], batch_size=1)

# -------------------------------
# Fonctions d'entraînement et test
# -------------------------------
def train():
    model.train()
    total_loss = 0
    for data in train_loader:
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out[train_mask], data.y[train_mask])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

def test():
    model.eval()
    correct = 0
    with torch.no_grad():
        for data in test_loader:
            out = model(data)
            pred = out.argmax(dim=1)
            correct += (pred[test_mask] == data.y[test_mask]).sum().item()
    return correct / test_mask.sum().item()

# -------------------------------
# Boucle d'entraînement
# -------------------------------
for epoch in range(N_EPOCHS):
    loss = train()
    acc = test()
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:03d} | Loss: {loss:.4f} | Accuracy: {acc:.4f}")

In [ ]:
# 12 | Predictions for every spot and the held-out accuracy (original cells 89, 91, 93)
graph_data1 = graph_data1.to(DEVICE); model = model.to(DEVICE)
import torch
from sklearn.metrics import accuracy_score, classification_report

# Passer le modèle en mode évaluation
model.eval()

# ⚡ Prédiction
with torch.no_grad():
    out = model(graph_data1)        # Appel correct : juste le Data object
    preds = out.argmax(dim=1)      # Classe prédite pour chaque cellule

# Labels réels
y_true = graph_data1.y

# Évaluation
acc = accuracy_score(y_true.cpu(), preds.cpu())
print(f"Accuracy: {acc:.4f}\n")

print("Classification report:")
print(classification_report(y_true.cpu(), preds.cpu()))

import torch
import pandas as pd

# Mettre le modèle en mode évaluation
model.eval()

# Définir l'appareil (GPU si disponible)
device = torch.device(DEVICE)
graph_data1 = graph_data1.to(device)
model = model.to(device)

# Pas de calcul du gradient pour l'inférence
with torch.no_grad():
    out = model(graph_data1)                  # logits pour chaque cellule
    preds = out.argmax(dim=1).cpu().numpy()  # classes prédites

# Récupérer les catégories des pseudo-labels
categories = pd.Categorical(adata.obs['pseudo_cell_type']).categories

# Convertir les indices en noms de classes
pred_labels = pd.Categorical.from_codes(preds, categories)

# Ajouter les prédictions dans adata.obs
adata.obs['predicted_cell_type'] = pred_labels

# Afficher le nombre de cellules par type prédit
print(adata.obs['predicted_cell_type'].value_counts())
TEST_ACCURACY = test()
print(f'held-out accuracy (20% of spots): {TEST_ACCURACY:.4f}')

In [ ]:
# 13 | Spatial images of the two 7-dpi infected capture areas (original cell 104, verbatim)
if 'spatial' in adata.uns:
    # Make a copy of the current spatial data
    spatial_data = adata.uns['spatial'].copy()
    
    # Swap the infected_7dpi_1 and infected_7dpi_2 entries
    if 'infected_7dpi_1' in spatial_data and 'infected_7dpi_2' in spatial_data:
        # Store the data temporarily
        temp_7dpi_1 = spatial_data['infected_7dpi_1']
        temp_7dpi_2 = spatial_data['infected_7dpi_2']
    
        # Swap them
        spatial_data['infected_7dpi_1'] = temp_7dpi_2
        spatial_data['infected_7dpi_2'] = temp_7dpi_1
    
        # Update the uns['spatial'] with corrected mapping
        adata.uns['spatial'] = spatial_data
    
        print("Successfully swapped infected_7dpi_1 and infected_7dpi_2 spatial data")
    else:
        print("Warning: Could not find both samples in spatial data")

In [ ]:
# 14 | Save the labelled object and the model (original cell 105; new file names)
adata.write(OUT_LABELS)
MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), MODEL_OUT)
print("written", OUT_LABELS, "and", MODEL_OUT)

In [ ]:
# 15 | GCN embedding before the classifier (original cells 109-110)
import torch
import numpy as np

# ---- fonction utilitaire : embeddings du dernier bloc GNN (avant classifier)
def get_gnn_embeddings(model, data):
    model.eval()
    with torch.no_grad():
        x, edge_index, edge_weight = data.x, data.edge_index, getattr(data, 'edge_weight', None)
        x = torch.relu(model.conv1(x, edge_index, edge_weight))
        x = torch.dropout(x, p=model.dropout, train=False)
        x = torch.relu(model.conv2(x, edge_index, edge_weight))
        x = torch.dropout(x, p=model.dropout, train=False)
        x = torch.relu(model.conv3(x, edge_index, edge_weight))
        # x = représentation finale (n_cells, hidden_dim)
        return x.cpu().numpy()

# ---- déplacer sur GPU si dispo (sécurité)
device = torch.device(DEVICE)
_ = model.to(device)
_ = graph_data1.to(device)

X_gnn_embed = get_gnn_embeddings(model, graph_data1)  # shape: [n_cells, hidden_dim]
adata.obsm['X_gnn_embed'] = X_gnn_embed
print('X_gnn_embed:', adata.obsm['X_gnn_embed'].shape)

In [ ]:
# 16 | Save the final object (original cells 118, 121; new file name). Use it as ANN_H5AD in the audit notebook.
if 'tda_diagrams' in adata.uns:
    del adata.uns['tda_diagrams']
for k in ("pseudo_cell_type", "predicted_cell_type", "tda_leiden"):
    adata.obs[k] = adata.obs[k].astype(str).astype("category")
adata.write(OUT_FINAL)
print("written", OUT_FINAL, adata.shape)

In [ ]:
# 17 | Summary and comparison with the previous PlantAG run
from sklearn.metrics import adjusted_rand_score
summary = {"spots": adata.n_obs, "types_predicted": int(adata.obs["predicted_cell_type"].nunique()),
           "held_out_accuracy": round(float(TEST_ACCURACY), 4), "persistence_image_pixels": int(adata.obsm["images_vec1"].shape[1])}
counts = pd.DataFrame({"new": adata.obs["predicted_cell_type"].value_counts()})
if PREVIOUS_ANN.exists():
    old = sc.read_h5ad(PREVIOUS_ANN, backed="r")
    common = adata.obs_names.intersection(old.obs_names)
    o, n = old.obs.loc[common, "predicted_cell_type"].astype(str), adata.obs.loc[common, "predicted_cell_type"].astype(str)
    summary.update({"spots_in_both_runs": len(common), "same_type_pct": round(100 * float((o == n).mean()), 1),
                    "adjusted_rand_index": round(float(adjusted_rand_score(o, n)), 3)})
    counts["previous_run_same_spots"] = o.value_counts()
display(pd.Series(summary)); display(counts.fillna(0).astype(int))
tag = f"_depth{THIN_TO_MEDIAN_UMIS}" if THIN_TO_MEDIAN_UMIS else ""
pd.Series(summary).to_csv(BASE / f"plantag_retrain_summary{tag}.csv"); counts.to_csv(BASE / f"plantag_retrain_type_counts{tag}.csv")
# type composition per slide (time point), this run and the comparison run (% of the slide's spots)
slide_of = lambda o: o["sample"].astype(str).str.extract(r"_(\d+dpi)_")[0]
comp = {"this_run": pd.crosstab(slide_of(adata.obs), adata.obs["predicted_cell_type"].astype(str), normalize="index") * 100}
if PREVIOUS_ANN.exists():
    comp["comparison_run"] = pd.crosstab(slide_of(old.obs), old.obs["predicted_cell_type"].astype(str), normalize="index") * 100
comp = pd.concat(comp, names=["run", "slide"]).fillna(0).round(1)
display(comp); comp.to_csv(BASE / f"plantag_retrain_composition_by_slide{tag}.csv")